In [1]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Raw Data 정리
# -----------------------------
models = ["Gemma-2-9B", "Llama-3.1-8B", "Qwen2.5-7B", "Mistral-7B"]

datasets = {
    "moral": {
        "framing": {
            "flip": [0.7030, 0.5030, 0.5680, 0.4615],
            "l1":   [1.0,    0.654384, 0.967157, 0.803942],
            "quad": [
                [0.7030, 0.0, 0.2970, 0.0],
                [0.4675, 0.0355, 0.1420, 0.3550],
                [0.5621, 0.0059, 0.0651, 0.3669],
                [0.4497, 0.0118, 0.0769, 0.4615]
            ]
        },
        "para": {
            "flip": [0.5743, 0.1302, 0.1657, 0.1598],
            "l1":   [1.0,    0.170934, 0.260410, 0.263490],
            "quad": [
                [0.5743, 0.0, 0.4257, 0.0],
                [0.0769, 0.0533, 0.1065, 0.7633],
                [0.1598, 0.0059, 0.0888, 0.7456],
                [0.1538, 0.0059, 0.1243, 0.7160]
            ]
        }
    },

    "scotus": {
        "framing": {
            "flip": [0.7030, 0.3441, 0.4355, 0.4086],
            "l1":   [1.0,    0.401569, 0.768808, 0.696127],
            "quad": [
                [0.7030, 0.0, 0.2970, 0.0],
                [0.2419, 0.1022, 0.2258, 0.4301],
                [0.4247, 0.0108, 0.2097, 0.3548],
                [0.4032, 0.0054, 0.1720, 0.4194]
            ]
        },
        "para": {
            "flip": [0.5743, 0.2043, 0.1613, 0.1989],
            "l1":   [1.0,    0.222367, 0.264786, 0.300725],
            "quad": [
                [0.5743, 0.0, 0.4257, 0.0],
                [0.1075, 0.0968, 0.1452, 0.6505],
                [0.1398, 0.0215, 0.1344, 0.7043],
                [0.1720, 0.0269, 0.1613, 0.6398]
            ]
        }
    },

    "triage": {
        "framing": {
            "flip": [0.7030, 0.5600, 0.5000, 0.5000],
            "l1":   [1.0,    0.499226, 0.836900, 0.921304],
            "quad": [
                [0.7030, 0.0, 0.2970, 0.0],
                [0.4200, 0.1400, 0.2400, 0.2000],
                [0.4800, 0.0200, 0.2200, 0.2800],
                [0.4800, 0.0200, 0.1400, 0.3600]
            ]
        },
        "para": {
            "flip": [0.5743, 0.3000, 0.3600, 0.3200],
            "l1":   [1.0,    0.256505, 0.517952, 0.513924],
            "quad": [
                [0.5743, 0.0, 0.4257, 0.0],
                [0.1400, 0.1600, 0.2000, 0.5000],
                [0.3400, 0.0200, 0.1000, 0.5400],
                [0.3000, 0.0200, 0.1400, 0.5400]
            ]
        }
    }
}

# -----------------------------
# 평균 계산
# -----------------------------
def compute_avg(dataset_key):
    f = datasets[dataset_key]["framing"]
    p = datasets[dataset_key]["para"]

    return {
        "flip_f": np.array(f["flip"]),
        "flip_p": np.array(p["flip"]),
        "l1_f":   np.array(f["l1"]),
        "l1_p":   np.array(p["l1"]),
        "quad_f": np.array(f["quad"]),
        "quad_p": np.array(p["quad"])
    }

avg = {}
for key in datasets:
    avg[key] = compute_avg(key)

# 전체 평균
total = {
    "flip_f": np.mean([avg[k]["flip_f"] for k in avg], axis=0),
    "flip_p": np.mean([avg[k]["flip_p"] for k in avg], axis=0),
    "l1_f":   np.mean([avg[k]["l1_f"] for k in avg], axis=0),
    "l1_p":   np.mean([avg[k]["l1_p"] for k in avg], axis=0),
    "quad_f": np.mean([avg[k]["quad_f"] for k in avg], axis=0),
    "quad_p": np.mean([avg[k]["quad_p"] for k in avg], axis=0)
}

# -----------------------------
# 2. Plot 함수들
# -----------------------------
def plot_flip(data, title):
    x = np.arange(len(models))
    w = 0.35

    plt.figure(figsize=(8,5))
    plt.bar(x - w/2, data["flip_f"], w, label="Framing")
    plt.bar(x + w/2, data["flip_p"], w, label="Paraphrase")

    plt.xticks(x, models, rotation=15)
    plt.ylabel("Flip Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_l1(data, title):
    x = np.arange(len(models))
    w = 0.35

    plt.figure(figsize=(8,5))
    plt.bar(x - w/2, data["l1_f"], w, label="Framing")
    plt.bar(x + w/2, data["l1_p"], w, label="Paraphrase")

    plt.xticks(x, models, rotation=15)
    plt.ylabel("Avg L1 Distance")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_quadrant(data, title, mode="framing"):
    quad = data["quad_f"] if mode == "framing" else data["quad_p"]

    labels = ["flip_high", "flip_low", "noflip_high", "noflip_low"]
    x = np.arange(len(models))

    plt.figure(figsize=(8,5))

    bottom = np.zeros(len(models))
    for i in range(4):
        plt.bar(x, quad[:, i], bottom=bottom, label=labels[i])
        bottom += quad[:, i]

    plt.xticks(x, models, rotation=15)
    plt.ylabel("Proportion")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


# -----------------------------
# 3. 실행 (각 dataset + total)
# -----------------------------
for name, data in avg.items():
    plot_flip(data, f"{name.upper()} — Flip Rate")
    plot_l1(data, f"{name.upper()} — Avg L1 Distance")
    plot_quadrant(data, f"{name.upper()} — Quadrant (Framing)", "framing")
    plot_quadrant(data, f"{name.upper()} — Quadrant (Paraphrase)", "para")

# 전체 평균
plot_flip(total, "TOTAL — Flip Rate")
plot_l1(total, "TOTAL — Avg L1 Distance")
plot_quadrant(total, "TOTAL — Quadrant (Framing)", "framing")
plot_quadrant(total, "TOTAL — Quadrant (Paraphrase)", "para")

ModuleNotFoundError: No module named 'matplotlib'